Notebook: 05_qt_measurement.ipynb

Purpose: Generate beat-level QT measurements and reliability evidence from raw ECG signals.

Inputs:
- raw ECG waveforms

Outputs:
- qt_measurements.parquet
- measurement_reliability.parquet

Forbidden inputs: signal quality evidence from 02_signal_quality.ipynb.

# 05 — QT Measurement Reliability

Compute QT intervals every beat and derive reliability metrics based on multi-lead agreement and repeatability.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from ecg_analytics.datasets import PTBXLDataset, LUDBDataset, NSTDBDataset, INCARTDataset, QTDBDataset
from ecg_analytics.qt.measurement import measure_qt_intervals

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)
data_dir = root_dir / 'data'

adapters = {
    'ptbxl': PTBXLDataset,
    'ludb': LUDBDataset,
    'nstdb': NSTDBDataset,
    'incart': INCARTDataset,
    'qtdb': QTDBDataset,
}

rows = []

for dataset_name, adapter_cls in adapters.items():
    dataset_path = data_dir / dataset_name
    if not dataset_path.exists():
        continue
    adapter = adapter_cls(data_dir=data_dir)
    try:
        record_names = adapter.list_records()
    except Exception:
        continue

    for record_name in record_names:
        try:
            record = adapter.load_record(record_name)
        except Exception:
            continue

        record_id = f'{dataset_name}/{record_name}'
        signal = np.asarray(record.signal, dtype=float)
        if signal.ndim == 1:
            signal = signal.reshape(-1, 1)
        lead_names = getattr(record, 'lead_names', []) or []
        fs = float(getattr(record, 'fs', np.nan))

        for lead_index, lead_name in enumerate(lead_names):
            lead_signal = signal[:, lead_index] if signal.shape[1] > lead_index else signal[:, 0]
            measurements = measure_qt_intervals(lead_signal, fs)
            for beat_idx, m in enumerate(measurements, start=1):
                rows.append({
                    'record_id': record_id,
                    'lead_id': str(lead_name).lower(),
                    'beat_id': beat_idx,
                    'qt_ms': float(m.qt_ms) if getattr(m, 'qt_ms', None) is not None else np.nan,
                    'qrs_onset_sample': int(m.q_onset) if getattr(m, 'q_onset', None) is not None else np.nan,
                    't_end_sample': int(m.t_end) if getattr(m, 't_end', None) is not None else np.nan,
                    'measurement_valid': bool(getattr(m, 'qt_ms', None) is not None),
                    'rr_ms': float(m.rr_ms) if getattr(m, 'rr_ms', None) is not None else np.nan,
                })

qt_df = pd.DataFrame(rows)
if qt_df.empty:
    qt_df = pd.DataFrame(columns=[
        'record_id','lead_id','beat_id','qt_ms','qrs_onset_sample','t_end_sample','measurement_valid','rr_ms',
    ])
qt_df['qt_ms'] = pd.to_numeric(qt_df['qt_ms'], errors='coerce')
qt_df['rr_ms'] = pd.to_numeric(qt_df['rr_ms'], errors='coerce')
qt_df['measurement_valid'] = qt_df['measurement_valid'].astype(bool)
qt_measurements = qt_df[['record_id','lead_id','beat_id','qt_ms','qrs_onset_sample','t_end_sample','measurement_valid']].copy()

if not qt_df.empty:
    grouped = qt_df.groupby(['record_id','beat_id'])
    median_qt = grouped['qt_ms'].transform('median')
    mean_qt = grouped['qt_ms'].transform('mean')
    qt_df['qt_variance_leads'] = grouped['qt_ms'].transform('var').fillna(0.0)
    qt_df['qt_variance_beats'] = qt_df.groupby(['record_id','lead_id'])['qt_ms'].transform('var').fillna(0.0)
    qt_df['lead_agreement_score'] = (1.0 - np.clip(np.abs(qt_df['qt_ms'] - median_qt) / (median_qt + 1e-6), 0.0, 1.0)).fillna(0.0)
    qt_df['beat_agreement_score'] = (1.0 - np.clip(np.abs(qt_df['qt_ms'] - mean_qt) / (mean_qt + 1e-6), 0.0, 1.0)).fillna(0.0)
    lead_count = qt_df.groupby(['record_id','beat_id'])['lead_id'].transform('nunique').fillna(1.0)
    qt_df['missing_lead_penalty'] = 1.0 - np.clip((12.0 - lead_count) / 12.0, 0.0, 1.0)
    repeatability = qt_df.groupby(['record_id','lead_id'])['qt_ms'].transform('std').fillna(0.0)
    qt_df['repeatability_score'] = (1.0 - np.clip(repeatability / (qt_df['qt_ms'].abs() + 1e-6), 0.0, 1.0)).fillna(0.0)
    qt_df['bsqi'] = (0.5 + 0.5 * (1.0 - np.clip(np.abs(qt_df['qt_ms'] - median_qt) / 150.0, 0.0, 1.0))).fillna(0.0)
    qt_df['wsqi'] = (0.5 + 0.5 * (1.0 - np.clip(np.abs(qt_df['qt_ms'] - mean_qt) / 150.0, 0.0, 1.0))).fillna(0.0)
    qt_df['internal_consistency_score'] = qt_df[['bsqi','wsqi','lead_agreement_score','beat_agreement_score','repeatability_score']].mean(axis=1)
    reliability_df = qt_df[['record_id','lead_id','beat_id','bsqi','wsqi','lead_agreement_score','beat_agreement_score','qt_variance_leads','qt_variance_beats','missing_lead_penalty','repeatability_score','internal_consistency_score']].copy()
else:
    reliability_df = pd.DataFrame(columns=[
        'record_id','lead_id','beat_id','bsqi','wsqi','lead_agreement_score','beat_agreement_score','qt_variance_leads','qt_variance_beats','missing_lead_penalty','repeatability_score','internal_consistency_score',
    ])

assert set(qt_measurements.columns) >= {'record_id','lead_id','beat_id','qt_ms','qrs_onset_sample','t_end_sample','measurement_valid'}
assert set(reliability_df.columns) >= {'record_id','lead_id','beat_id','bsqi','wsqi','lead_agreement_score','beat_agreement_score','qt_variance_leads','qt_variance_beats','missing_lead_penalty','repeatability_score','internal_consistency_score'}
assert qt_measurements[['record_id','lead_id','beat_id']].duplicated().sum() == 0
assert reliability_df[['record_id','lead_id','beat_id']].duplicated().sum() == 0

qt_measurements.to_parquet(artifacts_dir / 'qt_measurements.parquet', index=False)
reliability_df.to_parquet(artifacts_dir / 'measurement_reliability.parquet', index=False)
print('Wrote qt_measurements.parquet and measurement_reliability.parquet')
